# Pipeline_Prune
Export the winning pruned checkpoint → INT8 QDQ ONNX for STM32 deployment.

**What this notebook does:**
1. Loads the best pruned checkpoint from `Model_Pruning` (MobileNetV2 structured 10%)
2. Exports INT8 QDQ ONNX only — no FP32 export, the NPU only benefits from INT8
3. Saves test files so you can run on STM32 and compute accuracy
4. Prints a final comparison table: Baseline INT8 vs Pruned INT8

> Baseline INT8 results come from `Pipeline_Quantz` — point `BASELINE_LABELS_NPZ`
> and `BASELINE_INT8_OUTPUT_NPZ` at those already-saved files. No re-running needed.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

!pip -q install onnx onnxruntime onnxruntime-tools
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2
from utils.train   import setup_device, evaluate

device = setup_device(seed=41)

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

# ── Pruned model (winner from Model_Pruning) ──
CKPT    = f"{SAVE_DIR}/mobilenetv2_pruned_structured_10pct.pth"
OUT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports/pruned_structured_10pct")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Baseline results from Pipeline_Quantz (already on Drive, no re-run needed) ──
BASELINE_DIR           = Path("/content/drive/My Drive/stm32-thesis/exports/baseline")
BASELINE_LABELS_NPZ    = BASELINE_DIR / "test_labels.npz"   # labels saved in Pipeline_Quantz
BASELINE_INT8_OUTPUT_NPZ = BASELINE_DIR / "output_int8.npz" # STM32 output from Pipeline_Quantz run

In [ ]:
# ── Export & evaluation helpers ──────────────────────────────────────────

def export_onnx(model, path):
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    print(f"✅ FP32 ONNX exported (used only as INT8 input): {path}")


def save_test_files(model, test_loader, out_dir, n=200):
    inputs, logits_list, labels = [], [], []
    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(test_loader):
            if i >= n: break
            x = x.to(device)
            out = model(x)
            inputs.append(x.cpu().numpy().astype("float32")[0])
            logits_list.append(out.cpu().numpy().astype("float32")[0])
            labels.append(int(y.item()))
    inputs = np.stack(inputs)
    labels = np.array(labels, dtype="int32")
    logits = np.stack(logits_list)
    np.savez(out_dir / "test_input.npz",  input=inputs)
    np.savez(out_dir / "test_labels.npz", label=labels)
    np.savez(out_dir / "test_logits.npz", input=inputs, logits=logits)
    print(f"✅ Test files saved — {inputs.shape}  labels: {labels.shape}")
    return out_dir / "test_input.npz", out_dir / "test_labels.npz"


def save_calib_file(train_loader, path, n=200):
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n: break
            xs.append(x.numpy().astype("float32")[0])
    xs = np.stack(xs)
    np.savez(path, input=xs)
    print(f"✅ Calib data saved: {path}  {xs.shape}")


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data): return None
        out = {"input": self.data[self.i:self.i+1]}; self.i += 1; return out
    def rewind(self): self.i = 0


def quantize_int8(fp32_path, calib_path, int8_path):
    quantize_static(
        model_input=str(fp32_path), model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8, weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"✅ INT8 QDQ ONNX: {int8_path}")


def compute_stm32_accuracy(labels_npz, outputs_npz, key="c_outputs_1", num_classes=2):
    labels = np.load(labels_npz)["label"].astype("int64")
    raw    = np.load(outputs_npz)[key]
    if raw.size != len(labels) * num_classes:
        raise ValueError(f"Size mismatch: {raw.size} vs {len(labels)*num_classes}")
    preds = np.argmax(raw.reshape(len(labels), num_classes), axis=1)
    return (preds == labels).mean() * 100

In [ ]:
# ── Load dataset & pruned model ────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64)
test_loader              = get_test_loader(batch_size=1)

model = VWW_MobileNetV2().to(device)
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()

val_acc = evaluate(model, val_loader, device)
print(f"Pruned model val accuracy: {val_acc*100:.2f}%")

In [ ]:
# ── Export FP32 ONNX (intermediate — needed only as input for INT8 quantiser) ───
# We do NOT deploy FP32 on STM32. The NPU only runs INT8.
fp32_path = OUT_DIR / "model_fp32_temp.onnx"
export_onnx(model, fp32_path)

In [ ]:
# ── Save test files (input + labels for STM32 eval) ──────────────────
input_npz, labels_npz = save_test_files(model, test_loader, OUT_DIR, n=200)

In [ ]:
# ── Save calibration data (train split only) ─────────────────────────
calib_npz = OUT_DIR / "calib_train.npz"
save_calib_file(train_loader, calib_npz, n=200)

In [ ]:
# ── INT8 QDQ quantisation ────────────────────────────────────────────
int8_path = OUT_DIR / "model_int8_qdq.onnx"
quantize_int8(fp32_path, calib_npz, int8_path)

In [ ]:
# ── ONNX Runtime sanity check ────────────────────────────────────────
sess   = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
sample = np.load(input_npz)["input"][0:1]
out    = sess.run(["logits"], {"input": sample})[0]
print(f"✅ INT8 ONNX check — output shape: {out.shape}  logits: {out[0]}")

## Run on STM32

Run the INT8 model through STM32 AI CLI (same as you did in `Pipeline_Quantz`):

```bash
# INT8 pruned model
stedgeai validate -m exports/pruned_structured_10pct/model_int8_qdq.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/pruned_structured_10pct/test_input.npz
```

Then rename the output and come back to the next cell:
```bash
mv network_val_io.npz exports/pruned_structured_10pct/output_int8.npz
```

In [ ]:
# ── Final comparison: Baseline INT8 vs Pruned INT8 on STM32 ───────────────
# Run the STM32 CLI step above first, then execute this cell.

pruned_int8_output = OUT_DIR / "output_int8.npz"

baseline_acc = compute_stm32_accuracy(BASELINE_LABELS_NPZ, BASELINE_INT8_OUTPUT_NPZ)
pruned_acc   = compute_stm32_accuracy(labels_npz,          pruned_int8_output)

delta = pruned_acc - baseline_acc

print("\n" + "="*54)
print("STM32 ON-DEVICE RESULTS (INT8)")
print("="*54)
print(f"{'Model':<32} {'Accuracy':>10}")
print("-"*54)
print(f"{'MobileNetV2 baseline (no pruning)':<32} {baseline_acc:>9.2f}%")
print(f"{'MobileNetV2 pruned structured 10%':<32} {pruned_acc:>9.2f}%")
print("-"*54)
print(f"{'Delta':<32} {delta:>+9.2f}%")
print("="*54)

if delta >= 0:
    print(f"✅ Pruning held accuracy. Smaller model, same performance on STM32.")
else:
    print(f"⚠️  Pruning cost {abs(delta):.2f}% accuracy on-device.")